In [0]:
from pyspark.sql.types import StructType, StructField,StringType, IntegerType, DateType, TimestampType, FloatType 

import pyspark.sql.functions as F

In [0]:
catalog_name = 'ecommerce'

###Read and Vakidate Bronze brands table

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_brands");
df_bronze.show(10)

### Silver Clean Data Frame

In [0]:
df_silver = df_bronze.withColumn("brand_name",F.trim(F.col("brand_name")));
df_silver.limit(5).show()

In [0]:
df_silver = df_silver.withColumn("brand_code", F.regexp_replace(F.col("brand_code"),r"[^A-Za-z0-9]",""))
df_silver.limit(10).show()

In [0]:
df_silver = df_silver.select("category_code").distinct()
df_silver.show()

In [0]:
#Anomalies Dictionary
anomalies ={
    "GROCERY" : "GRCY",
    "BOOKS" : "BKS",
    "TOYS" : "TOY"
}

df_silver = df_silver.replace(anomalies, subset="category_code")


In [0]:
df_silver.select("category_code").distinct().show()

In [0]:
#Writing raw_data into the silver layer (catalog name : ecommerce, schema name: silver, table name : slv_brands)

df_silver.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema", "true")\
            .saveAsTable(f"{catalog_name}.silver.slv_brands")